# Imports

In [1]:
import librosa
import pandas as pd
from typing import Callable, Union, List
import os
from pathlib import Path
import sys
from pprint import pprint

from audio_dataset import RavdessRawData
from Preprocess import audio_to_waveform, trim_silence

# Functions

In [2]:
def extract_audio_statistics(
    audio_paths: List[Path],
    stat_func: Callable[[Path], dict]
) -> pd.DataFrame:
    """
    Generate a DataFrame with statistics for each audio file.

    Each row corresponds to an audio file.
    Columns include the file path and attributes returned by `stat_func`.

    Parameters:
    - audio_paths: List of audio file paths.
    - stat_func: A function that takes a path and returns a dict of stats.

    Returns:
    - pd.DataFrame with one row per file and one column per attribute.
    """
    records = []
    for path in audio_paths:
        stats = stat_func(path)
        stats["path"] = str(path)
        records.append(stats)
    return pd.DataFrame(records)


def no_silence_duration_stat(path: Path) -> dict:
    """
    function to extract duration of the no-silence part of an audio file. meaning the duration after trimming silence from the start and end of the audio file.
    """
    # Here you would implement the logic to get the duration of the audio file.
    waveform, sample_rate = audio_to_waveform(path)
    trimmed_waveform = trim_silence(waveform)
    duration = librosa.get_duration(y=trimmed_waveform, sr=sample_rate)
    return {"duration": duration} 

# Change Working Dir To the Project Working Dir

In [2]:
# change the dir to the grandparent directory of the current working directory

current_dir = Path(os.getcwd())
grandparent_dir = current_dir.parent.parent
os.chdir(grandparent_dir)

In [3]:
os.getcwd()  # Check the cwd has updated

'c:\\Users\\noams\\Python Projects\\Audio_processing_project'

# Data Examination

## recording's silence analysis

In [18]:
ravdess_raw_data = RavdessRawData()
audio_paths_with_labels = list(ravdess_raw_data.all_data)
audio_paths = [path for path, _ in audio_paths_with_labels]
ravdess_silenced_duraion = extract_audio_statistics(audio_paths, no_silence_duration_stat)
print(ravdess_silenced_duraion.head())  # Display the first few rows of the DataFrame

   duration                                               path
0     1.664  RAVDESS\original_data\Actor_18\03-01-03-02-02-...
1     1.952  RAVDESS\original_data\Actor_12\03-01-03-02-02-...
2     1.888  RAVDESS\original_data\Actor_08\03-01-07-01-02-...
3     1.344  RAVDESS\original_data\Actor_13\03-01-08-02-01-...
4     2.560  RAVDESS\original_data\Actor_03\03-01-06-02-01-...


In [21]:
# show statistics of the audio files
ravdess_silenced_duraion.describe()  # Display the statistics of the DataFrame

,duration
count,1440.000000
mean,1.732832
std,0.349538
min,0.864000
25%,1.504000
50%,1.664000
75%,1.920000
max,3.412437


## TCAV analysis

In [ ]:
# force re-import of tcav_demo to get the latest changes
import importlib
import tcav_demo
importlib.reload(tcav_demo)

from tcav_demo import get_tcav_per_sample


#? OPTION 1: create the df:
# df_merged = get_tcav_per_sample() # about 10 minutes to run

#? OPTION 2: load tcav_per_sample_with_attributes.csv
df_merged = pd.read_csv("tcav_per_sample_with_attributes.csv")

### df_merged since it's a merge of all_attributes.csv and tcav_per_sample.csv.

In [ ]:
display(df_merged.head(5))
display(df_merged.describe())

,path,true_label,predicted_label,predicted_probability,concept_name,layer_name,positive_sign_count,positive_magnitude
0,RAVDESS\original_data\Actor_23\03-01-02-02-02-...,calm,calm,0.999167,long_constant_thick,module3.blocks.0.conv2,0.0,-0.479544
1,RAVDESS\original_data\Actor_23\03-01-02-02-02-...,calm,calm,0.999167,long_dropping_flat_thick,module3.blocks.0.conv2,0.0,-2.054141
2,RAVDESS\original_data\Actor_23\03-01-02-02-02-...,calm,calm,0.999167,long_dropping_steep_thick,module3.blocks.0.conv2,0.0,-0.572685
3,RAVDESS\original_data\Actor_23\03-01-02-02-02-...,calm,calm,0.999167,long_dropping_steep_thin,module3.blocks.0.conv2,0.0,-1.140559
4,RAVDESS\original_data\Actor_23\03-01-02-02-02-...,calm,calm,0.999167,long_rising_flat_thick,module3.blocks.0.conv2,0.0,-0.252294


,predicted_probability,positive_sign_count,positive_magnitude
count,17280.000000,17280.000000,17280.000000
mean,0.944766,0.494618,-0.053511
std,0.137872,0.499986,1.188936
min,0.233158,0.000000,-4.889296
25%,0.994333,0.000000,-0.857114
50%,0.998819,0.000000,-0.015965
75%,0.999190,1.000000,0.788005
max,0.999960,1.000000,4.315073


## average among all

In [14]:
# load tcav_per_sample_with_attributes.csv
df_merged = pd.read_csv(Path(r'tcav_per_sample_with_attributes.csv'))

# drop rows whose true_label != predicted_label
df_merged = df_merged[df_merged['true_label'] == df_merged['predicted_label']]

# rename 'predicted_label' to 'label'
df_merged = df_merged.rename(columns={'predicted_label': 'label'})

# drop unnecessary columns
df_merged = df_merged.drop(columns=['true_label', 'predicted_probability', 'layer_name'])

display(df_merged.head(5))
display(df_merged.describe())

,path,label,concept_name,positive_percentage,magnitude
0,RAVDESS\original_data\Actor_23\03-01-02-02-02-...,calm,long_constant_thick,0.0,-0.479544
1,RAVDESS\original_data\Actor_23\03-01-02-02-02-...,calm,long_dropping_flat_thick,0.0,-2.054141
2,RAVDESS\original_data\Actor_23\03-01-02-02-02-...,calm,long_dropping_steep_thick,0.0,-0.572685
3,RAVDESS\original_data\Actor_23\03-01-02-02-02-...,calm,long_dropping_steep_thin,0.0,-1.140559
4,RAVDESS\original_data\Actor_23\03-01-02-02-02-...,calm,long_rising_flat_thick,0.0,-0.252294


,positive_percentage,magnitude
count,16008.000000,16008.000000
mean,0.496377,-0.047172
std,0.500002,1.176382
min,0.000000,-4.605803
25%,0.000000,-0.840683
50%,0.000000,-0.011525
75%,1.000000,0.787403
max,1.000000,4.315073


In [15]:
# get the average positive_percentage and magnitude per label and concept_name
df_groupby = df_merged.groupby(['label', 'concept_name']).agg({'positive_percentage': 'mean', 'magnitude': 'mean'}).reset_index()

df_groupby.head(20)
df_groupby.shape

(96, 4)

In [16]:
df_groupby

,label,concept_name,positive_percentage,magnitude
0,angry,long_constant_thick,0.759358,0.304354
1,angry,long_dropping_flat_thick,0.994652,1.734152
2,angry,long_dropping_steep_thick,0.080214,-0.517362
3,angry,long_dropping_steep_thin,0.994652,1.329085
4,angry,long_rising_flat_thick,0.000000,-0.912051
...,...,...,...,...
91,surprised,short_constant_thick,0.920904,0.587009
92,surprised,short_dropping_steep_thick,0.937853,0.601205
93,surprised,short_dropping_steep_thin,0.887006,0.618544
94,surprised,short_rising_steep_thick,0.994350,1.807985


In [ ]:
df_groupby = df_groupby[(df_groupby['positive_percentage'] >= 0.999) | (df_groupby['positive_percentage'] <= 0.001)]
df_groupby.shape
display(df_groupby)

,label,concept_name,positive_percentage,magnitude
4,angry,long_rising_flat_thick,0.0,-0.912051
24,disgust,long_constant_thick,1.0,1.242475
27,disgust,long_dropping_steep_thin,1.0,1.518234
31,disgust,short_constant_thick,1.0,0.869754
37,fearful,long_dropping_flat_thick,0.0,-1.656069
38,fearful,long_dropping_steep_thick,0.0,-1.627666
39,fearful,long_dropping_steep_thin,0.0,-1.857421
42,fearful,long_rising_steep_thin,0.0,-2.262024
43,fearful,short_constant_thick,0.0,-1.342050
44,fearful,short_dropping_steep_thick,0.0,-1.423237


In [22]:
# give all the rows with disgust and long_constant_thick sorted by magnitude
df_merged = df_merged[(df_merged['label'] == 'disgust') & (df_merged['concept_name'] == 'long_constant_thick')]
df_merged = df_merged.sort_values(by='magnitude', ascending=False)
display(df_merged.head(20))
df_merged.shape

,path,label,concept_name,positive_percentage,magnitude
16332,RAVDESS\original_data\Actor_07\03-01-07-01-02-...,disgust,long_constant_thick,1.0,3.202342
1356,RAVDESS\original_data\Actor_09\03-01-07-01-02-...,disgust,long_constant_thick,1.0,2.943556
12084,RAVDESS\original_data\Actor_02\03-01-07-01-01-...,disgust,long_constant_thick,1.0,2.838130
15828,RAVDESS\original_data\Actor_04\03-01-07-02-01-...,disgust,long_constant_thick,1.0,2.547365
972,RAVDESS\original_data\Actor_06\03-01-07-01-02-...,disgust,long_constant_thick,1.0,2.505548
6264,RAVDESS\original_data\Actor_24\03-01-07-02-01-...,disgust,long_constant_thick,1.0,2.488054
5244,RAVDESS\original_data\Actor_04\03-01-07-02-02-...,disgust,long_constant_thick,1.0,2.264716
3624,RAVDESS\original_data\Actor_24\03-01-07-02-02-...,disgust,long_constant_thick,1.0,2.220698
13560,RAVDESS\original_data\Actor_15\03-01-07-02-01-...,disgust,long_constant_thick,1.0,2.152404
8988,RAVDESS\original_data\Actor_17\03-01-07-01-02-...,disgust,long_constant_thick,1.0,2.127205


(180, 5)

In [ ]:
df_merged.head(20)

,path,label,concept_name,positive_percentage,magnitude
16332,RAVDESS\original_data\Actor_07\03-01-07-01-02-...,disgust,long_constant_thick,1.0,3.202342
1356,RAVDESS\original_data\Actor_09\03-01-07-01-02-...,disgust,long_constant_thick,1.0,2.943556
12084,RAVDESS\original_data\Actor_02\03-01-07-01-01-...,disgust,long_constant_thick,1.0,2.838130
15828,RAVDESS\original_data\Actor_04\03-01-07-02-01-...,disgust,long_constant_thick,1.0,2.547365
972,RAVDESS\original_data\Actor_06\03-01-07-01-02-...,disgust,long_constant_thick,1.0,2.505548
6264,RAVDESS\original_data\Actor_24\03-01-07-02-01-...,disgust,long_constant_thick,1.0,2.488054
5244,RAVDESS\original_data\Actor_04\03-01-07-02-02-...,disgust,long_constant_thick,1.0,2.264716
3624,RAVDESS\original_data\Actor_24\03-01-07-02-02-...,disgust,long_constant_thick,1.0,2.220698
13560,RAVDESS\original_data\Actor_15\03-01-07-02-01-...,disgust,long_constant_thick,1.0,2.152404
8988,RAVDESS\original_data\Actor_17\03-01-07-01-02-...,disgust,long_constant_thick,1.0,2.127205


## average among women

In [36]:
# load tcav_per_sample_with_attributes.csv
df_merged = pd.read_csv(Path(r'tcav_per_sample_with_attributes.csv'))

# drop rows whose true_label != predicted_label
df_merged = df_merged[df_merged['true_label'] == df_merged['predicted_label']]

# rename 'predicted_label' to 'label'
df_merged = df_merged.rename(columns={'predicted_label': 'label'})

# drop unnecessary columns
df_merged = df_merged.drop(columns=['true_label', 'predicted_probability', 'layer_name'])

df_merged.shape

(16008, 5)

In [37]:
# remove all rows with men (whose path ends at an odd number)
df_merged = df_merged[~df_merged["path"].str.extract(r"(\d)(?=\.\w+$)")[0].astype(float).mod(2).eq(1)]
df_merged.shape
df_merged

,path,label,concept_name,positive_percentage,magnitude
12,RAVDESS\original_data\Actor_02\03-01-07-01-02-...,disgust,long_constant_thick,1.0,0.963829
13,RAVDESS\original_data\Actor_02\03-01-07-01-02-...,disgust,long_dropping_flat_thick,1.0,1.250827
14,RAVDESS\original_data\Actor_02\03-01-07-01-02-...,disgust,long_dropping_steep_thick,1.0,0.375296
15,RAVDESS\original_data\Actor_02\03-01-07-01-02-...,disgust,long_dropping_steep_thin,1.0,1.414820
16,RAVDESS\original_data\Actor_02\03-01-07-01-02-...,disgust,long_rising_flat_thick,1.0,0.597090
...,...,...,...,...,...
17239,RAVDESS\original_data\Actor_12\03-01-07-01-01-...,disgust,short_constant_thick,1.0,1.064069
17240,RAVDESS\original_data\Actor_12\03-01-07-01-01-...,disgust,short_dropping_steep_thick,1.0,0.305240
17241,RAVDESS\original_data\Actor_12\03-01-07-01-01-...,disgust,short_dropping_steep_thin,1.0,0.456646
17242,RAVDESS\original_data\Actor_12\03-01-07-01-01-...,disgust,short_rising_steep_thick,1.0,1.241321


In [38]:
# get the average positive_percentage and magnitude per label and concept_name
df_groupby = df_merged.groupby(['label', 'concept_name']).agg({'positive_percentage': 'mean', 'magnitude': 'mean'}).reset_index()

df_groupby.head(20)
df_groupby.shape

(96, 4)

In [35]:
df_groupby = df_groupby[(df_groupby['positive_percentage'] >= 0.999) | (df_groupby['positive_percentage'] <= 0.001)]
df_groupby.shape
display(df_groupby)

,label,concept_name,positive_percentage,magnitude
4,angry,long_rising_flat_thick,0.0,-0.912051
24,disgust,long_constant_thick,1.0,1.242475
27,disgust,long_dropping_steep_thin,1.0,1.518234
31,disgust,short_constant_thick,1.0,0.869754
37,fearful,long_dropping_flat_thick,0.0,-1.656069
38,fearful,long_dropping_steep_thick,0.0,-1.627666
39,fearful,long_dropping_steep_thin,0.0,-1.857421
42,fearful,long_rising_steep_thin,0.0,-2.262024
43,fearful,short_constant_thick,0.0,-1.342050
44,fearful,short_dropping_steep_thick,0.0,-1.423237


In [39]:
# give all the rows with FEMALE disgust and long_constant_thick sorted by magnitude
df_merged = df_merged[(df_merged['label'] == 'disgust') & (df_merged['concept_name'] == 'long_constant_thick')]
df_merged = df_merged.sort_values(by='magnitude', ascending=False)
display(df_merged.head(20))
df_merged.shape

,path,label,concept_name,positive_percentage,magnitude
12084,RAVDESS\original_data\Actor_02\03-01-07-01-01-...,disgust,long_constant_thick,1.0,2.838130
15828,RAVDESS\original_data\Actor_04\03-01-07-02-01-...,disgust,long_constant_thick,1.0,2.547365
972,RAVDESS\original_data\Actor_06\03-01-07-01-02-...,disgust,long_constant_thick,1.0,2.505548
6264,RAVDESS\original_data\Actor_24\03-01-07-02-01-...,disgust,long_constant_thick,1.0,2.488054
5244,RAVDESS\original_data\Actor_04\03-01-07-02-02-...,disgust,long_constant_thick,1.0,2.264716
3624,RAVDESS\original_data\Actor_24\03-01-07-02-02-...,disgust,long_constant_thick,1.0,2.220698
16248,RAVDESS\original_data\Actor_02\03-01-07-02-02-...,disgust,long_constant_thick,1.0,2.054619
1740,RAVDESS\original_data\Actor_10\03-01-07-01-01-...,disgust,long_constant_thick,1.0,1.998577
12036,RAVDESS\original_data\Actor_12\03-01-07-01-02-...,disgust,long_constant_thick,1.0,1.973854
3024,RAVDESS\original_data\Actor_12\03-01-07-02-02-...,disgust,long_constant_thick,1.0,1.964595


(91, 5)